# W8: 매출 Sheets 연동

매출 데이터를 집계해 Google Sheets에 저장하고, 실패 시 CSV로 fallback 합니다.


In [ ]:
import io
import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for font_path in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(font_path)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}

    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        out = model.generate_content(prompt)
        return getattr(out, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

from pathlib import Path

uploaded = safe_upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    sales = pd.read_csv(io.StringIO(uploaded[fn].decode("utf-8")))
else:
    sales = pd.DataFrame({
        "일자": pd.date_range("2026-01-01", periods=7, freq="D"),
        "채널": ["스마트스토어", "쿠팡", "스마트스토어", "쿠팡", "스마트스토어", "쿠팡", "스마트스토어"],
        "매출": [120000, 98000, 135000, 101000, 128000, 90000, 111000],
        "주문수": [22, 16, 24, 18, 26, 15, 20]
    })

summary = sales.groupby("채널", as_index=False).agg({"매출": "sum", "주문수": "sum"})

output_path = "retail_sales_summary.csv"

def write_csv():
    summary.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"CSV 저장: {output_path}")

# Sheets fallback flow
try:
    import gspread
    from google.oauth2.service_account import Credentials
    creds_json = os.getenv("GOOGLE_SERVICE_ACCOUNT_JSON", "")
    if not creds_json:
        raise RuntimeError("GOOGLE_SERVICE_ACCOUNT_JSON 미설정")
    creds = Credentials.from_service_account_info(json.loads(creds_json), scopes=["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"])
    gc = gspread.authorize(creds)
    sh = gc.create("retail_sales_summary")
    ws = sh.sheet1
    ws.append_row(list(summary.columns))
    ws.append_rows(summary.values.tolist())
    print(f"Sheets 저장 완료: {sh.url}")
except Exception as e:
    print(f"Sheets 연동 실패(컬백 CSV): {e}")
    write_csv()

print(summary)
